# System Metrics Box Plots

This notebook creates box plots showing the distribution of metrics across different systems.
Each metric value corresponds to a unique context window, and we aggregate over all values.


In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from typing import Literal

## Utility Functions


In [ ]:
# ============================================================================
# DATA LOADING & PREPARATION
# ============================================================================


def load_and_combine_json_files(filepaths: list[str]) -> dict:
    """Load and combine multiple JSON files, filtering out NaN values."""

    def filter_nans(data: dict) -> dict:
        filtered = {}
        for key, value in data.items():
            if isinstance(value, dict):
                filtered[key] = filter_nans(value)
            elif isinstance(value, list):
                filtered_list = [v for v in value if not (isinstance(v, float) and math.isnan(v))]
                if filtered_list:
                    filtered[key] = filtered_list
            else:
                filtered[key] = value
        return filtered

    def merge_dicts(d1: dict, d2: dict) -> dict:
        merged = d1.copy()
        for key, value in d2.items():
            if key in merged:
                if isinstance(value, dict) and isinstance(merged[key], dict):
                    merged[key] = merge_dicts(merged[key], value)
                elif isinstance(value, list) and isinstance(merged[key], list):
                    merged[key] = merged[key] + value
                else:
                    merged[key] = value
            else:
                merged[key] = value
        return merged

    combined = {}
    for filepath in filepaths:
        with open(filepath) as f:
            data = filter_nans(json.load(f))
        combined = merge_dicts(combined, data)
    return combined


def get_largest_horizon(data: dict) -> str:
    """Get the largest prediction horizon."""
    return str(max(int(h) for h in data.keys()))


def get_system_max_horizons(data: dict) -> dict[str, int]:
    """Get maximum horizon for each system."""
    system_horizons = {}
    for horizon_str, systems in data.items():
        horizon_int = int(horizon_str)
        for system_name in systems:
            system_horizons[system_name] = max(system_horizons.get(system_name, 0), horizon_int)
    return system_horizons


def remove_outliers(
    data: np.ndarray,
    method: str = "iqr",
    iqr_multiplier: float = 1.5,
    z_threshold: float = 3.0,
    percentile_range: tuple = (1, 99),
) -> np.ndarray:
    """Remove outliers from data using various methods.

    Args:
        data: Input array
        method: 'iqr', 'z-score', 'percentile', or 'none'
        iqr_multiplier: Multiplier for IQR method (default 1.5)
        z_threshold: Threshold for z-score method (default 3.0)
        percentile_range: Percentile range for percentile method (default (1, 99))

    Returns:
        Array with outliers removed
    """
    if len(data) == 0 or method == "none":
        return data

    if method == "iqr":
        q1, q3 = np.percentile(data, [25, 75])
        iqr = q3 - q1
        lower_bound = q1 - iqr_multiplier * iqr
        upper_bound = q3 + iqr_multiplier * iqr
        mask = (data >= lower_bound) & (data <= upper_bound)
    elif method == "z-score":
        z_scores = np.abs((data - np.mean(data)) / np.std(data))
        mask = z_scores < z_threshold
    elif method == "percentile":
        lower, upper = np.percentile(data, percentile_range)
        mask = (data >= lower) & (data <= upper)
    else:
        raise ValueError(f"Unknown method: {method}. Use 'iqr', 'z-score', 'percentile', or 'none'")

    return data[mask]


def prepare_metric_data(
    data: dict,
    metric_name: str,
    prediction_horizon: str | None = None,
    use_max_horizons: bool = False,
    aggregate_all: bool = False,
    exclude_outliers: bool = False,
    outlier_method: Literal["iqr", "z-score", "percentile"] | None = "iqr",
    outlier_params: dict = None,
) -> tuple:
    """Unified data preparation with optional outlier removal.

    Args:
        data: Combined data dictionary
        metric_name: Name of the metric to extract
        prediction_horizon: Specific horizon to use (ignored if use_max_horizons=True)
        use_max_horizons: Use each system's maximum available horizon
        aggregate_all: Return single array with all values combined
        exclude_outliers: Whether to remove outliers
        outlier_method: Method for outlier detection ('iqr', 'z-score', 'percentile', 'none')
        outlier_params: Parameters for outlier method (e.g., {'iqr_multiplier': 1.5})

    Returns: (metric_arrays, system_names, horizons_used)
    """
    metric_arrays = []
    system_names = []
    horizons_used = []
    outlier_params = outlier_params or {}

    if use_max_horizons:
        system_max_horizons = get_system_max_horizons(data)
        for system_name in sorted(system_max_horizons.keys()):
            max_horizon = system_max_horizons[system_name]
            horizon_str = str(max_horizon)

            if horizon_str in data and system_name in data[horizon_str]:
                metrics = data[horizon_str][system_name]
                if metric_name in metrics and metrics[metric_name]:
                    values = np.array(metrics[metric_name])
                    if len(values) > 0:
                        # Remove outliers if requested
                        if exclude_outliers:
                            values = remove_outliers(values, method=outlier_method, **outlier_params)

                        if len(values) > 0:  # Check again after outlier removal
                            if aggregate_all:
                                metric_arrays.extend(values)
                            else:
                                metric_arrays.append(values)
                                system_names.append(system_name)
                                horizons_used.append(max_horizon)
    else:
        if prediction_horizon is None:
            prediction_horizon = get_largest_horizon(data)

        if prediction_horizon not in data:
            raise ValueError(f"Horizon '{prediction_horizon}' not found")

        for system_name in sorted(data[prediction_horizon].keys()):
            metrics = data[prediction_horizon][system_name]
            if metric_name in metrics and metrics[metric_name]:
                values = np.array(metrics[metric_name])
                if len(values) > 0:
                    # Remove outliers if requested
                    if exclude_outliers:
                        values = remove_outliers(values, method=outlier_method, **outlier_params)

                    if len(values) > 0:  # Check again after outlier removal
                        if aggregate_all:
                            metric_arrays.extend(values)
                        else:
                            metric_arrays.append(values)
                            system_names.append(system_name)

    if aggregate_all:
        return np.array(metric_arrays), None, None

    if not metric_arrays:
        raise ValueError(f"No valid data for metric '{metric_name}'")

    return metric_arrays, system_names, horizons_used if use_max_horizons else None


In [ ]:
# ============================================================================
# PLOTTING UTILITIES
# ============================================================================

METRIC_DISPLAY_NAMES = {
    "mase": "mase",
    "wape": "WAPE",
    "mase": "MASE",
    "wql": "WQL",
    "spearman": "Spearman",
    "pearson": "Pearson",
    "ssim": "SSIM",
    "energy_distance": "Energy Distance",
    "mmd": "MMD",
    "spectral_hellinger": "Spectral Hellinger",
    "spectral_wasserstein-1": "Spectral W-1",
    "spectral_wasserstein-2": "Spectral W-2",
    "cross_spectral_phase_similarity": "Cross-Spectral Phase Similarity",
}


def get_metric_display_name(metric_name: str) -> str:
    return METRIC_DISPLAY_NAMES.get(metric_name, metric_name)


def create_boxplot_core(
    ax: plt.Axes,
    data_arrays: list[np.ndarray],
    positions: np.ndarray,
    colors: list | str,
    percentile_range: tuple[float, float] = (25, 75),
    whisker_range: tuple[float, float] = (10, 90),
    show_mean_line: bool = True,
    box_width: float = 0.6,
) -> None:
    """Core box plot rendering logic."""
    if isinstance(colors, str):
        colors = [colors] * len(data_arrays)

    if percentile_range != (25, 75):
        # Custom percentiles - manual drawing
        for pos, arr, color in zip(positions, data_arrays, colors):
            stats = {
                "med": np.median(arr),
                "mean": np.mean(arr),
                "q1": np.percentile(arr, percentile_range[0]),
                "q3": np.percentile(arr, percentile_range[1]),
                "whislo": np.percentile(arr, whisker_range[0]),
                "whishi": np.percentile(arr, whisker_range[1]),
            }

            box = Rectangle(
                (pos - box_width / 2, stats["q1"]),
                box_width,
                stats["q3"] - stats["q1"],
                facecolor=color,
                edgecolor="black",
                linewidth=1,
                alpha=0.7,
            )
            ax.add_patch(box)

            ax.plot(
                [pos - box_width / 2, pos + box_width / 2], [stats["med"], stats["med"]], "k-", linewidth=2, zorder=3
            )
            if show_mean_line:
                ax.plot(
                    [pos - box_width / 2, pos + box_width / 2],
                    [stats["mean"], stats["mean"]],
                    "k--",
                    linewidth=1,
                    alpha=0.8,
                    zorder=3,
                )

            ax.plot([pos, pos], [stats["q3"], stats["whishi"]], "k-", linewidth=1, zorder=2)
            ax.plot([pos, pos], [stats["q1"], stats["whislo"]], "k-", linewidth=1, zorder=2)

            cap = box_width * 0.3
            ax.plot([pos - cap / 2, pos + cap / 2], [stats["whishi"], stats["whishi"]], "k-", linewidth=1)
            ax.plot([pos - cap / 2, pos + cap / 2], [stats["whislo"], stats["whislo"]], "k-", linewidth=1)
    else:
        # Standard matplotlib boxplot
        bp = ax.boxplot(
            data_arrays,
            positions=positions,
            widths=box_width,
            patch_artist=True,
            showmeans=True,
            meanline=True,
            showfliers=False,
            whis=whisker_range,
            medianprops=dict(color="black", linewidth=2),
            meanprops=dict(color="black", linewidth=1, linestyle="--") if show_mean_line else dict(alpha=0),
        )

        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
            patch.set_edgecolor("black")
            patch.set_linewidth(1)


def add_legend(ax: plt.Axes, show_mean: bool = True, horizon_colors: dict = None) -> None:
    """Add standardized legend."""
    elements = [Line2D([0], [0], color="black", linewidth=2, label="Median")]
    if show_mean:
        elements.append(Line2D([0], [0], color="black", linewidth=1, linestyle="--", label="Mean"))
    if horizon_colors:
        elements.append(Line2D([0], [0], color="none", label=""))
        for h in sorted(horizon_colors.keys(), reverse=True):
            elements.append(Patch(facecolor=horizon_colors[h], edgecolor="black", label=f"Horizon {h}", alpha=0.7))
    ax.legend(handles=elements, loc="upper right", frameon=True, framealpha=0.9)


def get_colors(n: int, cmap: str = "Blues", cmap_range: tuple = (0.3, 0.9)) -> list:
    """Generate n colors from colormap."""
    cmap_obj = plt.cm.get_cmap(cmap)
    return [cmap_obj(cmap_range[0] + (cmap_range[1] - cmap_range[0]) * i / max(1, n - 1)) for i in range(n)]


In [ ]:
# ============================================================================
# MAIN PLOTTING FUNCTIONS (with outlier exclusion support)
# ============================================================================


def plot_boxplots(
    data: dict,
    metric_name: str,
    prediction_horizon: str | None = None,
    use_max_horizons: bool = False,
    aggregate_all: bool = False,
    title: str | None = None,
    percentile_range: tuple[float, float] = (25, 75),
    whisker_range: tuple[float, float] = (10, 90),
    show_mean_line: bool = True,
    show_legend: bool = True,
    show_horizon_labels: bool = False,
    tick_rotation: int = 45,
    cmap: str = "Blues",
    color: str = "#3498db",
    show_stats: bool = False,
    exclude_outliers: bool = False,
    outlier_method: str = "iqr",
    outlier_params: dict = None,
    figsize: tuple[int, int] = (6, 8),
    save_path: str | None = None,
) -> plt.Figure:
    """Unified box plot function with outlier exclusion.

    Outlier Methods:
        - iqr: IQR method (default, multiplier=1.5)
        - z-score: Z-score method (threshold=3.0)
        - percentile: Percentile cutoff (range=(1, 99))
        - none: No outlier removal

    Examples:
        Without outlier removal: plot_boxplots(data, 'mase')
        With IQR removal: plot_boxplots(data, 'mase', exclude_outliers=True)
        Custom IQR: plot_boxplots(data, 'mase', exclude_outliers=True,
                                 outlier_params={'iqr_multiplier': 3.0})
        Z-score method: plot_boxplots(data, 'mase', exclude_outliers=True,
                                     outlier_method='z-score')
    """
    # Prepare data with optional outlier removal
    metric_arrays, system_names, horizons_used = prepare_metric_data(
        data,
        metric_name,
        prediction_horizon,
        use_max_horizons,
        aggregate_all,
        exclude_outliers,
        outlier_method,
        outlier_params,
    )

    metric_display = get_metric_display_name(metric_name)

    # Determine horizon label
    if use_max_horizons:
        horizon_label = "Max Horizons per System"
    else:
        if prediction_horizon is None:
            prediction_horizon = get_largest_horizon(data)
        horizon_label = f"Horizon {prediction_horizon}"

    # Add outlier info to title
    outlier_label = ""
    if exclude_outliers:
        outlier_label = f" [Outliers Removed: {outlier_method}]"

    # Setup figure
    if aggregate_all:
        fig, ax = plt.subplots(figsize=figsize)

        create_boxplot_core(ax, [metric_arrays], np.array([0]), color, percentile_range, whisker_range, show_mean_line)

        ax.set_xticks([0])
        ax.set_xticklabels(["All Systems\nAll Windows"], fontsize=12, fontweight="bold")

        if title is None:
            title = f"Aggregate {metric_display}\n({horizon_label}){outlier_label}"

        # Add stats box
        if show_stats:
            stats = {
                "n": len(metric_arrays),
                "mean": np.mean(metric_arrays),
                "med": np.median(metric_arrays),
                "std": np.std(metric_arrays),
                "min": np.min(metric_arrays),
                "max": np.max(metric_arrays),
                "q1": np.percentile(metric_arrays, 25),
                "q3": np.percentile(metric_arrays, 75),
            }
            stats_text = "\n".join([f"{k} = {v:,.3f}" if k != "n" else f"{k} = {v:,}" for k, v in stats.items()])
            ax.text(
                0.02,
                0.98,
                stats_text,
                transform=ax.transAxes,
                va="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray"),
                fontsize=10,
                fontfamily="monospace",
            )
    else:
        # Multiple boxes
        if figsize is None:
            figsize = (max(12, len(system_names) * 0.5), 6)
        fig, ax = plt.subplots(figsize=figsize)

        positions = np.arange(len(system_names))

        # Colors based on horizons if available
        if use_max_horizons and horizons_used:
            unique_horizons = sorted(set(horizons_used), reverse=True)
            horizon_colors = {h: c for h, c in zip(unique_horizons, get_colors(len(unique_horizons), cmap))}
            colors = [horizon_colors[h] for h in horizons_used]
        else:
            colors = get_colors(len(metric_arrays), cmap)
            horizon_colors = None

        create_boxplot_core(ax, metric_arrays, positions, colors, percentile_range, whisker_range, show_mean_line)

        # X-axis labels
        ax.set_xticks(positions)
        if show_horizon_labels and horizons_used:
            labels = [f"{name}\n(h={h})" for name, h in zip(system_names, horizons_used)]
        else:
            labels = system_names
        ax.set_xticklabels(labels, rotation=tick_rotation, ha="right")

        if title is None:
            if use_max_horizons:
                title = f"{metric_display} at Max Horizon per System{outlier_label}"
            else:
                title = f"{metric_display} Across Systems ({horizon_label}){outlier_label}"

        if show_legend:
            add_legend(ax, show_mean_line, horizon_colors if use_max_horizons else None)

    # Common formatting
    ax.set_ylabel(metric_display, fontsize=12, fontweight="bold")
    if not aggregate_all:
        ax.set_xlabel("System", fontsize=12, fontweight="bold")
    ax.set_title(title, fontsize=14, fontweight="bold", pad=20)
    ax.yaxis.grid(True, linestyle="--", alpha=0.3, zorder=0)
    ax.set_axisbelow(True)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"Saved: {save_path}")

    return fig


def plot_max_horizons_bar(
    data: dict,
    figsize: tuple[int, int] = (14, 6),
    cmap: str = "viridis",
    save_path: str | None = None,
) -> plt.Figure:
    """Bar chart of maximum prediction horizon per system."""
    system_horizons = get_system_max_horizons(data)
    sorted_systems = sorted(system_horizons.items(), key=lambda x: (-x[1], x[0]))
    names, horizons = zip(*sorted_systems)

    fig, ax = plt.subplots(figsize=figsize)

    unique_horizons = sorted(set(horizons), reverse=True)
    horizon_colors = {h: c for h, c in zip(unique_horizons, get_colors(len(unique_horizons), cmap))}
    colors = [horizon_colors[h] for h in horizons]

    positions = np.arange(len(names))
    ax.bar(positions, horizons, color=colors, alpha=0.8, edgecolor="black", linewidth=0.5)

    # Labels on bars
    for pos, h in zip(positions, horizons):
        ax.text(pos, h, str(h), ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_xlabel("System", fontsize=12, fontweight="bold")
    ax.set_ylabel("Maximum Prediction Horizon", fontsize=12, fontweight="bold")
    ax.set_title(f"Maximum Prediction Horizon by System (n={len(names)})", fontsize=14, fontweight="bold", pad=20)
    ax.set_xticks(positions)
    ax.set_xticklabels(names, rotation=45, ha="right")
    ax.yaxis.grid(True, linestyle="--", alpha=0.3, zorder=0)
    ax.set_axisbelow(True)
    ax.set_ylim(bottom=0, top=max(horizons) * 1.1)

    # Legend
    legend_elements = [
        Patch(facecolor=horizon_colors[h], edgecolor="black", label=f"Horizon {h}", alpha=0.8) for h in unique_horizons
    ]
    ax.legend(handles=legend_elements, loc="upper right", frameon=True, framealpha=0.9, title="Available Horizons")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig


## Load Data


In [ ]:
# File paths
base_path = Path(
    "../../outputs/ablations_timesfm_google-timesfm-2.5-200m-pytorch/rseed-123/original_vs_labels/all_heads"
)
json_files = [
    base_path / "metrics_long_gift-eval_nwindows-10_za-head_layergroup-1_all_heads.json",
    base_path / "metrics_short_gift-eval_nwindows-10_za-head_layergroup-1_all_heads.json",
]

# Load and combine
print("Loading...")
data = load_and_combine_json_files([str(f) for f in json_files])

# Summary
horizons = sorted(data.keys(), key=int)
print(f"Horizons: {horizons}")
print(f"Largest: {get_largest_horizon(data)}")


In [ ]:
# # Percentile Method (keep 1st to 99th percentile)
# plot_boxplots(
#     data,
#     "mase",
#     use_max_horizons=True,
#     exclude_outliers=True,
#     outlier_method="percentile",
#     outlier_params={"percentile_range": (5, 95)},
#     figsize=(16, 6),
# )
# plt.show()


# prepare_metric_data(
#     data, "mase", use_max_horizons=True, exclude_outliers=True, outlier_method="iqr"
# )


### Plot

In [ ]:
plot_boxplots(data, "mase", figsize=(16, 6))
plt.show()

In [ ]:
plot_boxplots(data, "mase", use_max_horizons=True, show_horizon_labels=True, figsize=(24, 8), cmap="viridis")
plt.show()

In [ ]:
plot_boxplots(
    data,
    "smape",
    use_max_horizons=True,
    # use_max_horizons=False,
    # prediction_horizon="64",
    aggregate_all=True,
    show_stats=True,
    exclude_outliers=True,
    outlier_method="percentile",
    outlier_params={"percentile_range": (5, 95)},
    figsize=(6, 6),
)
plt.show()

In [ ]:
plot_max_horizons_bar(data, figsize=(16, 6))
plt.show()

In [ ]:
metrics = ["smape", "wape", "mase", "wql"]
fig, axes = plt.subplots(1, len(metrics), figsize=(20, 6))

for ax, metric in zip(axes, metrics):
    values, _, _ = prepare_metric_data(
        data,
        metric,
        use_max_horizons=True,
        aggregate_all=True,
        exclude_outliers=True,
        outlier_method="iqr",
        outlier_params={"percentile_range": (5, 95)},
    )
    create_boxplot_core(ax, [values], np.array([0]), "#3498db")

    display_name = get_metric_display_name(metric)
    ax.set_ylabel(display_name, fontsize=12, fontweight="bold")
    ax.set_title(f"{display_name}\n(n={len(values):,})", fontsize=14, fontweight="bold")
    ax.set_xticks([0])
    ax.set_xticklabels(["All"])
    ax.yaxis.grid(True, linestyle="--", alpha=0.3, zorder=0)
    ax.set_axisbelow(True)

fig.suptitle("Aggregate Metric Distributions (Max Horizons)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
metric = "mase"
arrays, names, horizons = prepare_metric_data(data, metric, use_max_horizons=True)

print(f"Statistics for {metric} at max horizons:\n")
print(f"{'System':<40} {'H':>4} {'N':>5} {'Mean':>8} {'Med':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("=" * 100)

for name, arr, h in zip(names, arrays, horizons):
    print(
        f"{name:<40} {h:>4} {len(arr):>5} {np.mean(arr):>8.2f} {np.median(arr):>8.2f} "
        f"{np.std(arr):>8.2f} {np.min(arr):>8.2f} {np.max(arr):>8.2f}"
    )

# Overall aggregate
all_values, _, _ = prepare_metric_data(data, metric, use_max_horizons=True, aggregate_all=True)
print("\n" + "=" * 100)
print(
    f"{'AGGREGATE':<40} {'ALL':>4} {len(all_values):>5} {np.mean(all_values):>8.2f} {np.median(all_values):>8.2f} "
    f"{np.std(all_values):>8.2f} {np.min(all_values):>8.2f} {np.max(all_values):>8.2f}"
)